Load in libraries

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
import pandas as pd
from scipy.stats import chi2_contingency

Load in finalised data

In [11]:
# === Define file path ===
file_path = r"C:\Users\youri\OneDrive\Desktop\TIL Programming\6020 Group project\df_cleaned_population_area.csv"

# === Load CSV into DataFrame ===
df_merged = pd.read_csv(file_path)

Normalize network length

In [12]:
df_merged["km_per_million_people"] = df_merged["Network_length_KM"] / (df_merged["population"] / 1_000_000)
df_merged["km_per_1000_sqkm"] = df_merged["Network_length_KM"] / (df_merged["land_area_km2"] / 1000)

Check

In [13]:
# Display the entire DataFrame (be careful if it's large)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(df_merged)

,geo,TIME_PERIOD,Network_length_KM,Consignment_full_train_load_THS_T,Consignment_full_wagon_load_THS_T,Consignment_total_THS_T,uncertainty_estimate_NL,uncertainty_estimate_FULL_TR,uncertainty_estimate_FULL_WG,uncertainty_estimate_TOTAL,population,uncertainty_estimate_POP,land_area_km2,km_per_million_people,km_per_1000_sqkm
0,CH,2008,5146.000000,37705.000000,25631.000000,63336.000000,NaN,NaN,NaN,NaN,7.712250e+06,28266.381550,41289.0,667.250177,124.633680
1,CH,2009,5188.000000,35765.000000,20875.000000,56640.000000,NaN,NaN,NaN,NaN,7.787544e+06,25901.857604,41289.0,666.192031,125.650900
2,CH,2010,5124.000000,39125.000000,21293.000000,60418.000000,NaN,NaN,NaN,NaN,7.862839e+06,23558.913893,41289.0,651.673016,124.100850
3,CH,2011,5140.000000,40349.000000,20766.000000,61114.000000,NaN,NaN,NaN,NaN,7.938134e+06,21244.691464,41289.0,647.507367,124.488363
4,CH,2012,5147.000000,34013.000000,21593.000000,55607.000000,NaN,NaN,NaN,NaN,8.013428e+06,18969.704850,41289.0,642.296885,124.657899
5,CH,2013,5181.000000,36401.000000,23104.000000,59505.000000,NaN,NaN,NaN,NaN,8.088723e+06,16749.948774,41289.0,640.521388,125.481363
6,CH,2014,5187.000000,38197.000000,22149.000000,60345.000000,NaN,NaN,NaN,NaN,8.139631e+06,NaN,41289.0,637.252475,125.626680
7,CH,2015,5205.000000,39475.000000,21357.000000,60832.000000,NaN,NaN,NaN,NaN,8.237666e+06,NaN,41289.0,631.853731,126.062632
8,CH,2016,5249.000000,40040.000000,21108.000000,61148.000000,NaN,NaN,NaN,NaN,8.327126e+06,NaN,41289.0,630.349535,127.128291
9,CH,2017,5251.000000,44459.000000,14638.000000,59097.000000,NaN,NaN,NaN,NaN,8.419550e+06,NaN,41289.0,623.667536,127.176730


Saving DF to CSV

In [14]:
df_merged.to_csv(r"C:\Users\youri\OneDrive\Desktop\TIL Programming\6020 Group project\df_cleaned_population_area_normalized.csv", index=False, encoding="utf-8")

Plotting for all density levels (km_per_million_people)

In [15]:
# Melt the consignment columns into a long format for Plotly
cols_to_keep = ["km_per_million_people", "Consignment_full_train_load_THS_T", "Consignment_full_wagon_load_THS_T", "Consignment_total_THS_T"]

df_plot = df_merged[cols_to_keep].melt(id_vars="km_per_million_people", var_name="Consignment_Type", value_name="Consignment_Volume")

In [22]:
fig = px.scatter(df_plot, x="km_per_million_people", y="Consignment_Volume", color="Consignment_Type", trendline="ols", template="plotly_white", title="Relationship between network population density and consignment type")

fig.update_layout(
    title_font=dict(size=24),                  # Bigger title
    xaxis_title="Network population density (km/1*10^6 people)",
    yaxis_title="Consignment volume (thousand tonnes)",
    xaxis_title_font=dict(size=18),            # Bigger x-axis label
    yaxis_title_font=dict(size=18),            # Bigger y-axis label
    font=dict(size=16),                        # General font size (ticks, legend, etc.)
    legend_title="Consignment Type",
    legend_title_font=dict(size=18),
    legend_font=dict(size=16)
)

fig.show()
fig.write_image("network_pop_density_vs_consignments.png", width=1800, height=1000, scale=3)

In [ ]:
# Copy your main dataframe
df_chi = df_merged.copy()

# Create 3 density categories (Low / Medium / High) based on quantiles
df_chi["pop_density_group"] = pd.qcut(df_chi["km_per_million_people"], q=3, labels=["Low", "Medium", "High"])

# Determine dominant consignment type per observation (the one with the highest tonnage)
consignment_cols = ["Consignment_full_train_load_THS_T", "Consignment_full_wagon_load_THS_T", "Consignment_total_THS_T"]

df_chi["dominant_type"] = df_chi[consignment_cols].idxmax(axis=1)

# Simplify names for readability
df_chi["dominant_type"] = df_chi["dominant_type"].replace({"Consignment_full_train_load_THS_T": "Full Train", "Consignment_full_wagon_load_THS_T": "Full Wagon", "Consignment_total_THS_T": "Total"})

# Build the contingency table
contingency_pop = pd.crosstab(df_chi["pop_density_group"], df_chi["dominant_type"])
print("Population Density Contingency Table:\n", contingency_pop)

# Run Chi-squared test
chi2_pop, p_pop, dof_pop, expected_pop = chi2_contingency(contingency_pop)

print(f"\nChi-squared (Population Density): {chi2_pop:.3f}")
print(f"Degrees of freedom: {dof_pop}")
print(f"P-value: {p_pop:.4f}")


Population Density Contingency Table:
 dominant_type      Full Train  Total
pop_density_group                   
Low                         3     37
Medium                      0     39
High                        0     40

Chi-squared (Population Density): 6.078
Degrees of freedom: 2
P-value: 0.0479


Plotting for all density levels (km_per_1000sqkm)

In [17]:
# Melt the consignment columns into a long format for Plotly
cols_to_keep = ["km_per_1000_sqkm", "Consignment_full_train_load_THS_T", "Consignment_full_wagon_load_THS_T", "Consignment_total_THS_T"]

df_plot_2 = df_merged[cols_to_keep].melt(id_vars="km_per_1000_sqkm", var_name="Consignment_Type", value_name="Consignment_Volume")

In [ ]:
fig_2 = px.scatter(df_plot_2, x="km_per_1000_sqkm", y="Consignment_Volume", color="Consignment_Type", trendline="ols", template="plotly_white", title="Relationship between network area-density and consignment type")

fig_2.update_layout(title_font=dict(size=24), xaxis_title="Network length per area (km/1000 km²)", yaxis_title="Consignment volume (thousand tonnes)", xaxis_title_font=dict(size=18), yaxis_title_font=dict(size=18), font=dict(size=16), legend_title="Consignment Type", legend_title_font=dict(size=18), legend_font=dict(size=16))

fig_2.show()
fig_2.write_image("network_area_density_vs_consignments.png", width=1800, height=1000, scale=3)

In [ ]:
# Create 3 area density groups (Low / Medium / High)
df_chi["area_density_group"] = pd.qcut(df_chi["km_per_1000_sqkm"], q=3, labels=["Low", "Medium", "High"])

# Build contingency table for area density
contingency_area = pd.crosstab(df_chi["area_density_group"], df_chi["dominant_type"])
print("Area Density Contingency Table:\n", contingency_area)

# Run Chi-squared test
chi2_area, p_area, dof_area, expected_area = chi2_contingency(contingency_area)

print(f"\nChi-squared (Area Density): {chi2_area:.3f}")
print(f"Degrees of freedom: {dof_area}")
print(f"P-value: {p_area:.4f}")

Area Density Contingency Table:
 dominant_type       Full Train  Total
area_density_group                   
Low                          3     44
Medium                       0     33
High                         0     39

Chi-squared (Area Density): 4.715
Degrees of freedom: 2
P-value: 0.0947
